In [ ]:
!pip install segmentation_models_pytorch

#change, from best model val loss save to best model based on dice score 

In [ ]:
import numpy as np
import cv2
from tqdm import tqdm
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

import os, gc

#from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import Dataset, DataLoader,TensorDataset
from datasets import load_dataset



In [ ]:
dataset = load_dataset("RationAI/PanNuke")

print(dataset)

Testing distance map and category map for single sample

In [ ]:
fold = dataset["fold1"]
sample = fold[0]
image = np.array(sample["image"])
instance_mask = np.array(sample["instances"])
categories= np.array(sample["categories"])


In [ ]:
print(image.shape)
print(instance_mask.shape)
print(categories)

In [ ]:
print(instance_mask.shape)
print(np.unique(instance_mask))

In [ ]:
instance_map = np.zeros((256, 256), dtype=np.int32)
#creating unique id for each nucleus
for i in range(len(categories)):
    instance_map[instance_mask[i] > 0] = i + 1

In [ ]:
category_map = np.zeros((256, 256), dtype=np.int32)
#0 is for bg
for i in range(len(categories)):
    category_map[instance_mask[i] > 0] = categories[i]+1

In [ ]:
#over n dimention (each binary instance mask) check if more than one nucleus is occupying same pixel
overlap = np.sum(instance_mask, axis=0)
print(np.max(overlap))

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Image")
plt.imshow(image)

plt.subplot(1,3,2)
plt.title("Instance Map")
plt.imshow(instance_map)

plt.subplot(1,3,3)
plt.title("Category Map")
plt.imshow(category_map)

plt.show()

In [ ]:
import numpy as np
import pickle
import cv2
import torch
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==============================================================================
# PART 1: PREPROCESSING LOGIC
# ==============================================================================
def process_sample(sample):
    """Converts raw PanNuke sample into Multi-Task arrays."""
    image = np.array(sample["image"])
    instance_mask = np.array(sample["instances"]) 
    categories = np.array(sample["categories"])   

    H, W = image.shape[:2]

    # Initialize Task Maps
    instance_map = np.zeros((H, W), dtype=np.int32)
    category_map = np.zeros((H, W), dtype=np.int32)
    dist_map = np.zeros((H, W), dtype=np.float32)
    boundary_map = np.zeros((H, W), dtype=np.uint8)

    if len(instance_mask) == 0:
        return {
            "image": image.astype(np.uint8), # Keep uint8 for Albumentations
            "instance_map": instance_map,
            "category_map": category_map,
            "dist_map": dist_map,
            "boundary_map": boundary_map
        }

    for i in range(len(categories)):
        mask = (instance_mask[i] > 0).astype(np.uint8)
        
        # 1. Instance and Category Maps
        new_pixels = (mask > 0) & (instance_map == 0)
        instance_map[new_pixels] = i + 1
        category_map[new_pixels] = categories[i] + 1 # 0 becomes background

        # 2. Distance Map (Normalized per nucleus)
        if np.any(mask):
            individual_dist = distance_transform_edt(mask)
            if individual_dist.max() > 0:
                individual_dist = individual_dist / individual_dist.max()
            dist_map = np.maximum(dist_map, individual_dist)

        # 3. Boundary Map (1-pixel wide contour)
        kernel = np.ones((3,3), np.uint8)
        erosion = cv2.erode(mask, kernel, iterations=1)
        boundary = mask - erosion
        boundary_map = np.maximum(boundary_map, boundary)

    return {
        "image": image.astype(np.uint8),
        "instance_map": instance_map,
        "category_map": category_map,
        "dist_map": dist_map.astype(np.float32),
        "boundary_map": boundary_map.astype(np.uint8),
    }

def process_and_save_dataset(dataset_dict, save_path="/kaggle/working/processed_pannuke.pkl"):
    """Processes folds and saves to disk."""
    processed_folds = {}
    for fold_name, fold_data in dataset_dict.items():
        print(f"\n[INFO] Processing {fold_name}...")
        processed_samples = []
        for idx in tqdm(range(len(fold_data))):
            try:
                processed = process_sample(fold_data[idx])
                processed_samples.append(processed)
            except Exception as e:
                print(f"[WARNING] Skipping sample {idx} in {fold_name}: {e}")
        processed_folds[fold_name] = processed_samples

    with open(save_path, "wb") as f:
        pickle.dump(processed_folds, f)
    print(f"\n[INFO] Dataset saved successfully to {save_path}")



In [ ]:
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
# ==============================================================================
# PART 2: AUGMENTATIONS & NORMALIZATION (ALBUMENTATIONS)
# ==============================================================================
# Tell Albumentations to treat these custom targets like spatial masks
target_mapping = {
    'category_map': 'mask',
    'dist_map': 'mask',
    'boundary_map': 'mask'
}

# Get the preprocessing parameters for efficientnet-b1
preprocessing_fn = smp.encoders.get_preprocessing_fn('efficientnet-b1', pretrained='imagenet')
train_transform = A.Compose([
    # Spatial transforms (affects image and all masks)
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ElasticTransform(alpha=1, sigma=50,  p=0.2),
    
    # Pixel-level transforms (affects ONLY the image)
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.3),
    A.Lambda(image=preprocessing_fn),
    # Normalize and convert to PyTorch Tensor format
    #A.ToFloat(max_value=255.0),
    ToTensorV2() 
], additional_targets=target_mapping)

val_transform = A.Compose([A.Lambda(image=preprocessing_fn),
    # No spatial/color augmentations, just normalization and tensor conversion
    #A.ToFloat(max_value=255.0),
    ToTensorV2()
], additional_targets=target_mapping)




In [ ]:
class PanNukeMultiTaskUNet(nn.Module):
    def __init__(self, encoder_name='efficientnet-b1', encoder_weights='imagenet'):
        super().__init__()
        self.shared_unet = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=16,
            activation=None
        )
        self.category_head = nn.Conv2d(16, 6, kernel_size=1)
        self.dist_head = nn.Conv2d(16, 1, kernel_size=1)
        self.boundary_head = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        shared_features = self.shared_unet(x)
        category_logits = self.category_head(shared_features)
        
        # APPLY SIGMOID HERE to fix the 0.0 Dice issue
        dist_map = torch.sigmoid(self.dist_head(shared_features))
        
        boundary_logits = self.boundary_head(shared_features)
        return category_logits, dist_map, boundary_logits

In [ ]:
# ==============================================================================
# PART 3: PYTORCH DATASET CLASS
# ==============================================================================
class PanNukeDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        
        image = sample["image"]
        instance_map = sample["instance_map"]
        category_map = sample["category_map"]
        dist_map = sample["dist_map"]
        boundary_map = sample["boundary_map"]

        if self.transform:
            augmented = self.transform(
                image=image,
                mask=instance_map, # Required base mask parameter
                category_map=category_map,
                dist_map=dist_map,
                boundary_map=boundary_map
            )
            
            image = augmented["image"]
            # We don't strictly need instance_map for training loss, but useful for debugging
            instance_map = augmented["mask"] 
            category_map = augmented["category_map"]
            dist_map = augmented["dist_map"]
            boundary_map = augmented["boundary_map"]

        # Ensure correct PyTorch dtypes for their respective loss functions
        category_map = category_map.to(torch.long)    # For CrossEntropyLoss
        dist_map = dist_map.to(torch.float32)         # For MSELoss
        boundary_map = boundary_map.to(torch.float32) # For BCEWithLogitsLoss
        
        # We unsqueeze distance and boundary to add a channel dimension [1, H, W]
        # This matches the output shape of the U-Net Conv2d heads
        dist_map = dist_map.unsqueeze(0)
        boundary_map = boundary_map.unsqueeze(0)

        return image, category_map, dist_map, boundary_map


In [ ]:
process_and_save_dataset(dataset)

In [ ]:
#visualising preproceesed maps
import pickle
import matplotlib.pyplot as plt
import random
import os

# 1. Create the output directory
save_folder = "/kaggle/working/processes_labels"
os.makedirs(save_folder, exist_ok=True)
print(f"[INFO] Created/Verified save directory: {save_folder}")

# 2. Load the processed data dictionary
print("[INFO] Loading saved dataset...")

with open("/kaggle/working/processed_pannuke.pkl", "rb") as f:
    processed_folds = pickle.load(f)

# 3. Grab just the first fold for a quick visual test
fold_names = list(processed_folds.keys())
# test_fold_name = fold_names[0]
# print(f"[INFO] Testing with data from: {test_fold_name}")
# test_data = processed_folds[test_fold_name]

# # 4. Instantiate the PyTorch Dataset using your training transform
# test_dataset = PanNukeDataset(test_data, transform=train_transform)

# 5. Visualization & Saving Function
def visualize_dataset_sample(dataset, num_samples=3, save_dir=None):
    print(f"[INFO] Plotting {num_samples} random augmented samples...")
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    
    # Handle edge case if num_samples is 1
    if num_samples == 1: 
        axes = [axes]

    # Keep track of the random indices to create a unique filename
    plotted_indices = []

    for i in range(num_samples):
        # Pick a random index from the dataset
        idx = random.randint(0, len(dataset) - 1)
        plotted_indices.append(str(idx))
        
        # Pull the data (this triggers the __getitem__ and augmentations)
        image, category_map, dist_map, boundary_map = dataset[idx]

        # Convert PyTorch Tensors back to Numpy arrays for Matplotlib
        img_np = image.permute(1, 2, 0).numpy() # [C, H, W] -> [H, W, C]
        cat_np = category_map.numpy()           # [H, W]
        dist_np = dist_map.squeeze(0).numpy()   # [1, H, W] -> [H, W]
        bound_np = boundary_map.squeeze(0).numpy() # [1, H, W] -> [H, W]

        # Plot H&E Image
        axes[i][0].imshow(img_np)
        axes[i][0].set_title(f"Sample {idx}: H&E Image")
        axes[i][0].axis("off")

        # Plot Category Map
        axes[i][1].imshow(cat_np, cmap="tab10", vmin=0, vmax=5)
        axes[i][1].set_title("Categories (0=BG)")
        axes[i][1].axis("off")

        # Plot Distance Map
        axes[i][2].imshow(dist_np, cmap="magma")
        axes[i][2].set_title("Distance Map")
        axes[i][2].axis("off")

        # Plot Boundary Map
        axes[i][3].imshow(bound_np, cmap="gray")
        axes[i][3].set_title("Boundary Mask")
        axes[i][3].axis("off")

    plt.tight_layout()
    
    # --- SAVING LOGIC ---
    if save_dir:
        # Create a filename like: "augmented_samples_142_59_811.png"
        filename = f"augmented_samples_{'_'.join(plotted_indices)}.png"
        filepath = os.path.join(save_dir, filename)
        
        # Save the figure. dpi=150 ensures good resolution. bbox_inches='tight' removes extra white space.
        plt.savefig(filepath, dpi=150, bbox_inches='tight')
        print(f"[INFO] Successfully saved visualization grid to: {filepath}")

    # Show the plot in the notebook
    plt.show()

# 6. Execute the visualization and save the output
#visualize_dataset_sample(test_dataset, num_samples=3, save_dir=save_folder)

In [ ]:
# ==============================================================================
# PART 4: EXECUTION EXAMPLE (HOW TO RUN IT)
# ==============================================================================
"""
# 1. Assume `raw_dataset_dict` is your HuggingFace/loaded raw dictionary
# process_and_save_dataset(raw_dataset_dict)

# 2. Load the processed data
with open("/kaggle/working/processed_pannuke.pkl", "rb") as f:
    processed_folds = pickle.load(f)

# 3. Create splits (Example: Fold 1 & 2 for Train, Fold 3 for Validation)
train_data = processed_folds['Fold 1'] + processed_folds['Fold 2']
val_data = processed_folds['Fold 3']

# 4. Instantiate Datasets
train_dataset = PanNukeDataset(train_data, transform=train_transform)
val_dataset = PanNukeDataset(val_data, transform=val_transform)

# 5. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# Quick check to ensure it works
img, cat, dist, bound = next(iter(train_loader))
print(f"Image batch shape: {img.shape}")
print(f"Category batch shape: {cat.shape}")
print(f"Distance batch shape: {dist.shape}")
print(f"Boundary batch shape: {bound.shape}")
"""

In [ ]:
import os

# Let's see exactly what Kaggle thinks is in your folder
print("Files in working directory:", os.listdir('/kaggle/working/'))

# Try opening using a relative path instead of absolute
file_path = "processed_pannuke.pkl" 

if os.path.exists(file_path):
    print(f"✅ Found it! Loading {file_path}...")
    with open(file_path, "rb") as f:
        processed_folds = pickle.load(f)
else:
    print(f"❌ Still can't see {file_path}. Check the spelling in the sidebar!")

In [ ]:
with open("/kaggle/working/processed_pannuke.pkl", "rb") as f:
    processed_folds = pickle.load(f)

In [ ]:
# This will print the exact keys maintained in your saved file
print("Maintained Folds:", processed_folds.keys())

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
'''
# ==============================================================================
# 1. MULTI-TASK U-NET ARCHITECTURE
# ==============================================================================
class PanNukeMultiTaskUNet(nn.Module):
    def __init__(self, encoder_name='resnet34', encoder_weights='imagenet'):
        super().__init__()
        
        # Shared Encoder/Decoder (Outputs 16 generic features)
        self.shared_unet = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,      
            classes=16,         
            activation=None     
        )
        
        # Task-Specific Heads
        self.category_head = nn.Conv2d(16, 6, kernel_size=1) # 6 classes
        self.dist_head = nn.Conv2d(16, 1, kernel_size=1)     # 1 channel (regression)
        self.boundary_head = nn.Conv2d(16, 1, kernel_size=1) # 1 channel (binary)

    def forward(self, x):
        shared_features = self.shared_unet(x)
        
        category_logits = self.category_head(shared_features)
        dist_logits = self.dist_head(shared_features)
        boundary_logits = self.boundary_head(shared_features)
        
        return category_logits, dist_logits, boundary_logits
'''
# ==============================================================================
# 2. THE COMBINED LOSS FUNCTION
# ==============================================================================
class PanNukeLoss(nn.Module):
    def __init__(self, weight_cat=1.0, weight_dist=5.0, weight_bound=1.0):
        super().__init__()
        self.w_cat = weight_cat
        self.w_dist = weight_dist
        self.w_bound = weight_bound
        
        self.ce_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()
        self.bce_loss = nn.BCEWithLogitsLoss()

    def dice_loss(self, pred, target, smooth=1e-7):
        # pred is already sigmoid-activated from the updated forward pass
        iflat = pred.contiguous().view(-1)
        tflat = target.contiguous().view(-1)
        intersection = (iflat * tflat).sum()
        
        return 1 - ((2. * intersection + smooth) / (iflat.sum() + tflat.sum() + smooth))

    def forward(self, preds, targets):
        pred_cat, pred_dist, pred_bound = preds
        target_cat, target_dist, target_bound = targets

        # 1. Classification Loss
        loss_cat = self.ce_loss(pred_cat, target_cat)
        
        # 2. Hybrid Distance Loss (MSE for regression + Dice for overlap)
        # Using both ensures the model learns the distance values AND the mask shape
        loss_mse = self.mse_loss(pred_dist, target_dist)
        loss_dice = self.dice_loss(pred_dist, target_dist)
        loss_dist = loss_mse + loss_dice
        
        # 3. Boundary Loss
        loss_bound = self.bce_loss(pred_bound, target_bound)

        # Total Weighted Loss
        total_loss = (self.w_cat * loss_cat) + \
                     (self.w_dist * loss_dist) + \
                     (self.w_bound * loss_bound)

        return total_loss, loss_cat, loss_dist, loss_bound

In [ ]:
# ==============================================================================
# 3. QUICK SANITY CHECK (TESTING THE FLOW)
# ==============================================================================
if __name__ == "__main__":
    print("[INFO] Initializing Model and Loss...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = PanNukeMultiTaskUNet().to(device)
    criterion = PanNukeLoss().to(device)
    
    # Grab a real batch from your train_loader (assuming it's loaded in memory)
    # image, cat_map, dist_map, bound_map = next(iter(train_loader))
    
    # Using dummy data just to prove it doesn't crash
    dummy_img = torch.randn(2, 3, 256, 256).to(device)
    dummy_cat = torch.randint(0, 6, (2, 256, 256)).to(device)
    dummy_dist = torch.rand(2, 1, 256, 256).to(device)
    dummy_bound = torch.rand(2, 1, 256, 256).to(device)
    
    # Forward Pass
    preds = model(dummy_img)
    targets = (dummy_cat, dummy_dist, dummy_bound)
    
    # Loss Calculation
    total_loss, l_cat, l_dist, l_bound = criterion(preds, targets)
    
    print(f"Total Loss: {total_loss.item():.4f}")
    print(f"  - Category Loss: {l_cat.item():.4f}")
    print(f"  - Distance Loss: {l_dist.item():.4f}")
    print(f"  - Boundary Loss: {l_bound.item():.4f}")
    print("[SUCCESS] Model and Loss function are fully compatible.")

In [ ]:

"""

# ==============================================================================
# HYPERPARAMETERS & SETUP
# ==============================================================================
NUM_EPOCHS = 10 # Start small to test, then increase to 30-50 for final run
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "/kaggle/working/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"[INFO] Using Device: {DEVICE}")

# fold_names and processed_folds should already be in memory from earlier steps
fold_names = list(processed_folds.keys())

# ==============================================================================
# CROSS-VALIDATION LOOP
# ==============================================================================
for val_fold in fold_names:
    print(f"\n" + "="*60)
    print(f" STARTING FOLD: Validation on {val_fold}")
    print("="*60)
    
    # 1. Prepare Data Split
    train_data = []
    for fold in fold_names:
        if fold == val_fold:
            val_data = processed_folds[fold]
        else:
            train_data.extend(processed_folds[fold])

    # Instantiate Datasets and DataLoaders
    train_dataset = PanNukeDataset(train_data, transform=train_transform)
    val_dataset = PanNukeDataset(val_data, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

    # 2. Initialize Model, Loss, and Optimizer (FRESH FOR EACH FOLD)
    model = PanNukeMultiTaskUNet(encoder_name='resnet34', encoder_weights='imagenet').to(DEVICE)
    criterion = PanNukeLoss(weight_cat=1.0, weight_dist=1.0, weight_bound=1.0).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Mixed Precision Scaler (Speeds up training on modern GPUs)
    scaler = torch.cuda.amp.GradScaler()

    best_val_loss = float('inf')

    # 3. Epoch Loop
    for epoch in range(NUM_EPOCHS):
        # ---------------------------------------------------------
        # TRAINING PHASE
        # ---------------------------------------------------------
        model.train()
        train_loss = 0.0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)
        for images, targets_cat, targets_dist, targets_bound in loop:
            images = images.to(DEVICE)
            targets_cat = targets_cat.to(DEVICE)
            targets_dist = targets_dist.to(DEVICE)
            targets_bound = targets_bound.to(DEVICE)

            optimizer.zero_grad()

            # Forward pass with Mixed Precision
            with torch.cuda.amp.autocast():
                preds = model(images)
                targets = (targets_cat, targets_dist, targets_bound)
                loss, l_cat, l_dist, l_bound = criterion(preds, targets)

            # Backward pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)

        # ---------------------------------------------------------
        # VALIDATION PHASE
        # ---------------------------------------------------------
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad(): # No gradients needed for validation
            for images, targets_cat, targets_dist, targets_bound in val_loader:
                images = images.to(DEVICE)
                targets_cat = targets_cat.to(DEVICE)
                targets_dist = targets_dist.to(DEVICE)
                targets_bound = targets_bound.to(DEVICE)

                with torch.cuda.amp.autocast():
                    preds = model(images)
                    targets = (targets_cat, targets_dist, targets_bound)
                    loss, _, _, _ = criterion(preds, targets)

                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        # Print Epoch Summary
        print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # ---------------------------------------------------------
        # SAVE BEST MODEL
        # ---------------------------------------------------------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_path = os.path.join(SAVE_DIR, f"best_model_val_{val_fold}.pth")
            torch.save(model.state_dict(), save_path)
            print(f"  🌟 Validation loss improved! Model saved to {save_path}")

    print(f"✅ Finished training fold. Best Val Loss: {best_val_loss:.4f}\n")

"""

In [ ]:
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from sklearn.metrics import f1_score, jaccard_score

# ==============================================================================
# 1. DEFINE POST-PROCESSING (The missing function)
# ==============================================================================
def compute_instances(dist_map):
    """
    Converts a distance map prediction into individual object instances 
    using local maxima and watershed.
    """
    dist_map = dist_map.astype(np.float32)
    # Threshold to create a mask of the nuclei
    mask = dist_map > 0.1
    
    # Find local peaks (centers of nuclei)
    coords = peak_local_max(dist_map, min_distance=7, labels=mask)
    
    # Create markers for watershed
    markers = np.zeros_like(dist_map, dtype=int)
    for i, (r, c) in enumerate(coords):
        markers[r, c] = i + 1
        
    # Separate touching nuclei
    return watershed(-dist_map, markers, mask=mask)

# ==============================================================================
# 2. DEFINE METRICS
# ==============================================================================
def calculate_metrics(pred_cat, target_cat, pred_inst, target_inst):
    """
    Calculates Dice and other metrics for the current image.
    """
    # Binary masks for pixel-level Dice
    true_bin = (target_inst > 0).astype(np.uint8)
    pred_bin = (pred_inst > 0).astype(np.uint8)
    
    if true_bin.sum() == 0: 
        return None 

    # Flatten for Dice/IoU
    t_flat = true_bin.flatten()
    p_flat = pred_bin.flatten()
    
    intersection = np.logical_and(t_flat, p_flat).sum()
    dice = (2. * intersection) / (t_flat.sum() + p_flat.sum() + 1e-7)
    
    return {"Dice": dice}

In [ ]:
import torch
import torch.optim as optim
import os
import numpy as np
from tqdm import tqdm

# ==============================================================================
# HYPERPARAMETERS & SETUP
# ==============================================================================
NUM_EPOCHS = 50  
EARLY_STOPPING_PATIENCE = 15  
LEARNING_RATE = 5e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "/kaggle/working/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

for val_fold in fold_names:
    print(f"\n🚀 STARTING FOLD: {val_fold}")
    
    # 1. Data Split (Standard logic)
    train_data = []
    for fold in fold_names:
        if fold == val_fold: val_data = processed_folds[fold]
        else: train_data.extend(processed_folds[fold])

    train_loader = DataLoader(PanNukeDataset(train_data, transform=train_transform), 
                              batch_size=8, shuffle=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(PanNukeDataset(val_data, transform=val_transform), 
                            batch_size=8, shuffle=False, num_workers=2)

    # 2. Model & Optimizer
    model = PanNukeMultiTaskUNet().to(DEVICE)
    criterion = PanNukeLoss().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    # NEW AMP SYNTAX
    scaler = torch.amp.GradScaler('cuda')

    best_val_dice = 0.0
    epochs_no_improve = 0 

    for epoch in range(NUM_EPOCHS):
        # --- TRAINING ---
        model.train()
        
    
        train_loss = 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False)
        
        for images, t_cat, t_dist, t_bound in loop:
            images = images.to(DEVICE).float()
            t_cat = t_cat.to(DEVICE)
            t_dist = t_dist.to(DEVICE).float() # Distance maps should also be float
            t_bound = t_bound.to(DEVICE).float()
            
            

            optimizer.zero_grad()
            
            # NEW AMP SYNTAX
            with torch.amp.autocast('cuda'):
                preds = model(images) # returns (cat, dist, bound)
                loss, _, _, _ = criterion(preds, (t_cat, t_dist, t_bound))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        # --- VALIDATION ---
        model.eval()
        val_dice_list = []
        
        with torch.no_grad():
            for images, _, _, t_inst in val_loader:
                images = images.to(DEVICE).float() 
                
                
                with torch.amp.autocast('cuda'):
                    # preds[1] is the distance map which now has Sigmoid
                    _, p_dist_tensor, _ = model(images)
                
                # Move batch to CPU for Watershed
                p_dist_batch = p_dist_tensor.detach().float().squeeze(1).cpu().numpy()
                t_inst_batch = t_inst.numpy()
                
                # Process each image in batch individually
                for i in range(p_dist_batch.shape[0]):
                    pd = p_dist_batch[i]
                    ti = t_inst_batch[i].squeeze()
                    
                    pi = compute_instances(pd)
                    res = calculate_metrics(None, None, pi, ti)
                    if res: val_dice_list.append(res["Dice"])

        avg_val_dice = np.mean(val_dice_list) if val_dice_list else 0.0
        scheduler.step(avg_val_dice)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss/len(train_loader):.4f} | Val Dice: {avg_val_dice:.4f} | LR: {current_lr}")

        # Best Model & Early Stopping
        if avg_val_dice > best_val_dice:
            best_val_dice = avg_val_dice
            epochs_no_improve = 0
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"best_model_{val_fold}.pth"))
            print("🌟 New Best Dice!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                print("🛑 Early Stopping.")
                break